# Compiling Algol (semi-detached) systems 
## from Malkov 2020

based on 
### "Semidetached double-lined eclipsing binaries: Stellar parameters and rare classes "
Malkov 2020
 https://ui.adsabs.harvard.edu/abs/2020MNRAS.491.5489M/abstract



*** 
From Malkov (2020MNRAS.491.5489M): 

The well-known Catalogue of Algol-Type Binary Stars \cite{2004A&A...417..263B} lists 411 Semi detached stars, however, the majority of those stars are drawn from the \cite{1980AcA....30..501B} and \cite{2004yCat.5124....0S} catalogues which only contain approximate data. Lastly \cite{2004yCat.5115....0S} provide parameters for 96 semi-detached binaries.

\cite{2020MNRAS.491.5489M} makes a new comprehensive list of semi-detached binaries with reliable absolute parameters containing 119 semidetached double-lined eclipsing binaries containing the orbital parameters and physical parameters of the components.



In [7]:
import json
from urllib.parse import quote

import numpy as np
import astropy.units as u
from astroquery.vizier import Vizier
import re


# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR

## Build json table from Simbad 

In [8]:
VIZIER_CAT = "J/MNRAS/491/5489/tablea1"
MALKOV_BIB = "2020MNRAS.491.5489M"

def triplet(val, err=None):
    """Return [err-, value, err+] with None for missing."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return [None, None, None]
    v = float(val)
    if err is None or (isinstance(err, float) and np.isnan(err)):
        return [None, v, None]
    e = float(err)
    return [e, v, e]

def simbad_url_from_coords(ra_deg, dec_deg, radius_arcsec=5):
    if ra_deg is None or dec_deg is None:
        return None
    coords = f"{float(ra_deg)} {float(dec_deg)}"
    return (
        "https://simbad.cds.unistra.fr/simbad/sim-coo?"
        f"Coord={quote(coords)}&Radius={radius_arcsec}&Radius.unit=arcsec&output.format=ASCII"
    )


def spectral_type_to_evol_type(sptype: str | None) -> str | None:
    if not sptype:
        return None

    s = sptype.strip().upper()

    # Compact objects (explicit)
    if "WD" in s:
        return "WD"
    if "NS" in s:
        return "NS"
    if "BH" in s:
        return "BH"

    # Helium / stripped stars
    if any(x in s for x in ["SDO", "SDB", "HE", "WR", "WN", "WC"]):
        return "He-star"

    # Luminosity class-based mapping
    # Order matters!
    if re.search(r"\bIV\b", s):
        return "HG"
    if re.search(r"\bIII\b", s) or re.search(r"\bII\b", s):
        return "RGB"
    if re.search(r"\bI(A|B)?\b", s):
        return "AGB"

    # Default: normal hydrogen-burning star
    return "MS"



# Pull catalog
viz = Vizier(columns=["*"])
viz.ROW_LIMIT = -1  # IMPORTANT: set on the instance
tbl = viz.get_catalogs(VIZIER_CAT)[0]


print("N rows:", len(tbl))
print("Columns:", tbl.colnames)

out = []
for row in tbl:
    name = str(row["GCVS"]).strip() if "GCVS" in row.colnames else None

    ra = float(row["_RA"]) if "_RA" in row.colnames and row["_RA"] is not None else None
    dec = float(row["_DE"]) if "_DE" in row.colnames and row["_DE"] is not None else None

    per = float(row["Per"]) if "Per" in row.colnames and row["Per"] is not None else None

    m1 = row["M1"] if "M1" in row.colnames else None
    e_m1 = row["e_M1"] if "e_M1" in row.colnames else None

    m2 = row["M2"] if "M2" in row.colnames else None
    e_m2 = row["e_M2"] if "e_M2" in row.colnames else None

    # Extract spectral types
    sptype = str(row["SpType"]).strip() if "SpType" in row.colnames and row["SpType"] is not None else None
    obs_type_1 = None
    obs_type_2 = None
    if sptype:
        if "+" in sptype:
            parts = [p.strip() for p in sptype.split("+", 1)]
            obs_type_1 = parts[0] if len(parts) > 0 else None
            obs_type_2 = parts[1] if len(parts) > 1 else None
        else:
            obs_type_1 = sptype

    bib = str(row["BibCode"]).strip() if "BibCode" in row.colnames and row["BibCode"] is not None else None
    refs = [MALKOV_BIB] + ([bib] if bib and bib != "None" else [])
    # de-dup while keeping order
    refs = list(dict.fromkeys(refs))

    entry = {
        "System Name": name,
        "RA": triplet(ra),
        "Dec": triplet(dec),

        "Period": triplet(per),
        "Eccentricity": [None, None, None],  # not provided in this VizieR table
        "M1": triplet(m1, e_m1),
        "M1_sin3i": [None, None, None],
        "M2": triplet(m2, e_m2),
        "M2_sin3i": [None, None, None],
        "q": [None, None, None],
        "Mass Function": [None, None, None],

        "obs_type_1": obs_type_1,
        "obs_type_2": obs_type_2,

        "evol_type_1": spectral_type_to_evol_type(obs_type_1),
        "evol_type_2": spectral_type_to_evol_type(obs_type_2),


        "system_class": "Algol",
        "Detection Method": ["EB", "SB2", "RV"],
        "Reference": refs,
        "Notes": "Semi-detached double-lined eclipsing binary compiled by Malkov (2020).",
        "Simbad": simbad_url_from_coords(ra, dec),
    }

    out.append(entry)

out_path = RAW_JSON_DIR / "Algols_Malkov2020.raw.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    f.write("[\n")
    for i, system in enumerate(out):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        f.write("  " + line)
        if i < len(out) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("]\n")

print(f"Wrote {len(out)} systems to {out_path}")


N rows: 119
Columns: ['GCVS', 'Per', 'SpType', 'M1', 'e_M1', 'M2', 'e_M2', 'R1', 'e_R1', 'R2', 'e_R2', 'T1', 'e_T1', 'T2', 'e_T2', 'logL1', 'e_logL1', 'logL2', 'e_logL2', 'a', 'e_a', 'BibCode', 'Simbad', '_RA', '_DE']
Wrote 119 systems to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/Algols_Malkov2020.raw.json


/var/folders/5d/vcxrsh5975l7d5n8t7vvc5pr0000gn/T/ipykernel_64502/3199617332.py:11: UserWarning: Warning: converting a masked element to nan.
  e = float(err)


### Inspect table before continuing

In [9]:
display(out)


[{'System Name': 'TW And',
  'RA': [None, 0.82595, None],
  'Dec': [None, 32.84586, None],
  'Period': [None, 4.12276035, None],
  'Eccentricity': [None, None, None],
  'M1': [nan, 1.684999942779541, nan],
  'M1_sin3i': [None, None, None],
  'M2': [nan, 0.32499998807907104, nan],
  'M2_sin3i': [None, None, None],
  'q': [None, None, None],
  'Mass Function': [None, None, None],
  'obs_type_1': 'FV',
  'obs_type_2': 'KIV',
  'evol_type_1': 'MS',
  'evol_type_2': 'MS',
  'system_class': 'Algol',
  'Detection Method': ['EB', 'SB2', 'RV'],
  'Reference': ['2020MNRAS.491.5489M', '2014AN....335.1064M'],
  'Notes': 'Semi-detached double-lined eclipsing binary compiled by Malkov (2020).',
  'Simbad': 'https://simbad.cds.unistra.fr/simbad/sim-coo?Coord=0.82595%2032.84586&Radius=5&Radius.unit=arcsec&output.format=ASCII'},
 {'System Name': 'WW And',
  'RA': [None, 356.22311, None],
  'Dec': [None, 45.68651, None],
  'Period': [None, 23.28525, None],
  'Eccentricity': [None, None, None],
  'M1': [